In [1]:
!git clone https://github.com/HKhangg/DS201_DeepLearning.1.git

Cloning into 'DS201_DeepLearning.1'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 196 (delta 61), reused 191 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 17.11 MiB | 19.66 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [2]:
%cd DS201_DeepLearning.1/Lab5

/kaggle/working/DS201_DeepLearning.1/Lab5


In [3]:
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.5 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

In [4]:
import sys
import os
import torch
import json
sys.path.append('src')

from models.transformer_encoder import (
    TransformerForSequenceClassification,
    TransformerForTokenClassification
)
from data.uit_viocd_dataset import create_uit_viocd_dataloaders
from data.phonert_dataset import create_phonert_dataloaders
from training.trainer import create_trainer
from utils.utils import set_seed, get_device, print_model_info

set_seed(42)
device = get_device()

Using GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.06 GB


## Bài 1

In [9]:
# Paths
train_path_viocd = 'src/data/UIT_ViOCD/train_preprocessed.json'
dev_path_viocd = 'src/data/UIT_ViOCD/dev_preprocessed.json'
test_path_viocd = 'src/data/UIT_ViOCD/test_preprocessed.json'

# Hyperparameters
BATCH_SIZE = 32
MAX_LEN = 128
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 3
D_FF = 1024
DROPOUT = 0.1
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
PATIENCE = 5

# Create dataloaders
print("Creating UIT-ViOCD dataloaders...")
train_loader_viocd, dev_loader_viocd, test_loader_viocd, vocab_viocd, num_classes = create_uit_viocd_dataloaders(
    train_path_viocd, dev_path_viocd, test_path_viocd,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    num_workers=2
)

print(f"\nVocabulary size: {len(vocab_viocd)}")
print(f"Number of classes: {num_classes}")
print(f"Train batches: {len(train_loader_viocd)}")
print(f"Dev batches: {len(dev_loader_viocd)}")
print(f"Test batches: {len(test_loader_viocd)}")

Creating UIT-ViOCD dataloaders...

Vocabulary size: 2670
Number of classes: 4
Train batches: 138
Dev batches: 18
Test batches: 18


In [10]:
# Create model
model_viocd = TransformerForSequenceClassification(
    vocab_size=len(vocab_viocd),
    num_classes=num_classes,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT,
    pad_idx=vocab_viocd.PAD_IDX
)

print_model_info(model_viocd)


MODEL INFORMATION
Total parameters: 3,053,828
Model size: 11.65 MB



In [11]:
# Create trainer
trainer_viocd = create_trainer(
    model=model_viocd,
    train_loader=train_loader_viocd,
    dev_loader=dev_loader_viocd,
    test_loader=test_loader_viocd,
    learning_rate=LEARNING_RATE,
    device=device,
    save_dir='checkpoints_viocd',
    task_type='classification'
)

# Train
trainer_viocd.train(num_epochs=NUM_EPOCHS, patience=PATIENCE)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training for 20 epochs...
Device: cuda
Task type: classification

Epoch 1/20


Evaluating: 100%|██████████| 18/18 [00:00<00:00, 85.31it/s]



Train Loss: 0.7846 | Train Accuracy: 0.4269
Dev Loss: 0.6243 | Dev Accuracy: 0.4653
Dev Precision: 0.4645 | Dev Recall: 0.4515 | Dev F1: 0.3539
Learning Rate: 0.000100
✓ Saved best model with Accuracy: 0.4653

Epoch 2/20


Evaluating: 100%|██████████| 18/18 [00:00<00:00, 85.67it/s]


...


Train Loss: 0.1731 | Train Accuracy: 0.5949
Dev Loss: 0.4542 | Dev Accuracy: 0.5474
Dev Precision: 0.3913 | Dev Recall: 0.6250 | Dev F1: 0.4729
Learning Rate: 0.000050

Epoch 15/20


Evaluating: 100%|██████████| 18/18 [00:00<00:00, 79.90it/s]



Train Loss: 0.1306 | Train Accuracy: 0.6020
Dev Loss: 0.5241 | Dev Accuracy: 0.5493
Dev Precision: 0.3921 | Dev Recall: 0.6240 | Dev F1: 0.4737
Learning Rate: 0.000050

Epoch 16/20


Evaluating: 100%|██████████| 18/18 [00:00<00:00, 87.08it/s]


Train Loss: 0.1125 | Train Accuracy: 0.6052
Dev Loss: 0.5731 | Dev Accuracy: 0.5511
Dev Precision: 0.4032 | Dev Recall: 0.6266 | Dev F1: 0.4785
Learning Rate: 0.000050

Early stopping triggered after 16 epochs

Training completed!
Best Accuracy: 0.5547 at epoch 11
Saved history to checkpoints_viocd/history.json


## Bài 2

In [12]:
# Paths
train_path_phonert = 'src/data/PhoNERT/train.json'
dev_path_phonert = 'src/data/PhoNERT/dev.json'
test_path_phonert = 'src/data/PhoNERT/test.json'

# Hyperparameters (giữ nguyên như Bài 1)
# Có thể điều chỉnh nếu cần

# Create dataloaders
print("Creating PhoNERT dataloaders...")
train_loader_phonert, dev_loader_phonert, test_loader_phonert, vocab_phonert, label_encoder, num_labels = create_phonert_dataloaders(
    train_path_phonert, dev_path_phonert, test_path_phonert,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    num_workers=2
)

print(f"\nVocabulary size: {len(vocab_phonert)}")
print(f"Number of labels: {num_labels}")
print(f"Label mapping: {label_encoder.label2idx}")
print(f"Train batches: {len(train_loader_phonert)}")
print(f"Dev batches: {len(dev_loader_phonert)}")
print(f"Test batches: {len(test_loader_phonert)}")

Creating PhoNERT dataloaders...

Vocabulary size: 3303
Number of labels: 21
Label mapping: {'<PAD>': 0, 'B-AGE': 1, 'B-DATE': 2, 'B-GENDER': 3, 'B-JOB': 4, 'B-LOCATION': 5, 'B-NAME': 6, 'B-ORGANIZATION': 7, 'B-PATIENT_ID': 8, 'B-SYMPTOM_AND_DISEASE': 9, 'B-TRANSPORTATION': 10, 'I-AGE': 11, 'I-DATE': 12, 'I-JOB': 13, 'I-LOCATION': 14, 'I-NAME': 15, 'I-ORGANIZATION': 16, 'I-PATIENT_ID': 17, 'I-SYMPTOM_AND_DISEASE': 18, 'I-TRANSPORTATION': 19, 'O': 20}
Train batches: 158
Dev batches: 63
Test batches: 94


In [13]:
# Create model
model_phonert = TransformerForTokenClassification(
    vocab_size=len(vocab_phonert),
    num_labels=num_labels,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT,
    pad_idx=vocab_phonert.PAD_IDX
)

print_model_info(model_phonert)


MODEL INFORMATION
Total parameters: 3,220,245
Model size: 12.28 MB



In [14]:
# Create trainer
trainer_phonert = create_trainer(
    model=model_phonert,
    train_loader=train_loader_phonert,
    dev_loader=dev_loader_phonert,
    test_loader=test_loader_phonert,
    learning_rate=LEARNING_RATE,
    device=device,
    save_dir='checkpoints_phonert',
    task_type='token_classification'
)

# Train
trainer_phonert.train(num_epochs=NUM_EPOCHS, patience=PATIENCE)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training for 20 epochs...
Device: cuda
Task type: token_classification

Epoch 1/20


Evaluating: 100%|██████████| 63/63 [00:00<00:00, 115.49it/s]



Train Loss: 0.7756 | Train F1-Score: 0.1699
Dev Loss: 0.6389 | Dev F1-Score: 0.2794
Dev Precision: 0.4309 | Dev Recall: 0.2445 | Dev F1: 0.2794
Learning Rate: 0.000100
✓ Saved best model with F1-Score: 0.2794

Epoch 2/20


Evaluating: 100%|██████████| 63/63 [00:00<00:00, 118.98it/s]


...


Train Loss: 0.1255 | Train F1-Score: 0.7230
Dev Loss: 0.2670 | Dev F1-Score: 0.6514
Dev Precision: 0.7070 | Dev Recall: 0.6307 | Dev F1: 0.6514
Learning Rate: 0.000100
✓ Saved best model with F1-Score: 0.6514

Epoch 19/20


Evaluating: 100%|██████████| 63/63 [00:00<00:00, 118.12it/s]



Train Loss: 0.1188 | Train F1-Score: 0.7400
Dev Loss: 0.2718 | Dev F1-Score: 0.6526
Dev Precision: 0.7086 | Dev Recall: 0.6306 | Dev F1: 0.6526
Learning Rate: 0.000100
✓ Saved best model with F1-Score: 0.6526

Epoch 20/20


Evaluating: 100%|██████████| 63/63 [00:00<00:00, 118.81it/s]



Train Loss: 0.1148 | Train F1-Score: 0.7458
Dev Loss: 0.2683 | Dev F1-Score: 0.6645
Dev Precision: 0.7211 | Dev Recall: 0.6395 | Dev F1: 0.6645
Learning Rate: 0.000100
✓ Saved best model with F1-Score: 0.6645

Training completed!
Best F1-Score: 0.6645 at epoch 20
Saved history to checkpoints_phonert/history.json
